# Vendi-RAG on a toy corpus

This notebook runs end to end on a laptop in under a minute. **No API key, no
model download, no network.** Everything below is computed here.

The argument it makes is the paper's:

> Similarity search optimizes each document in isolation, so what comes back is
> several restatements of the same fact. Multi-hop questions need the opposite —
> a few documents that say *different* things. Scoring the retrieved *set*, and
> steering how hard you push on that, is what closes the gap.

Paper: [arXiv:2502.11228](https://arxiv.org/abs/2502.11228)

---
### A word on what these numbers are

The corpus is synthetic and built to exhibit the redundancy failure mode, so
the effect sizes here are a property of this corpus, not a benchmark result.
They are reproducible — the corpus and the embedder are both deterministic —
and they show the mechanism working. For real-data numbers see the paper, or
`research/` to rerun them.

In [ ]:
# pip install vendirag        # the library needs only numpy
# pip install "vendirag[viz]"  # matplotlib + pillow, for the plots below

import numpy as np
import matplotlib.pyplot as plt

from vendirag import (
    HashingEmbedder, VendiRAG, VendiRetriever,
    mmr_select, vendi_score,
)
from vendirag.toy import make_corpus

K = 8      # documents retrieved per query
POOL = 50  # candidate pool size |C|

## 1. The corpus

A fictional world of Arctic expeditions. Each *chain* states four facts, one per
document, each phrased in the vocabulary of its own hop:

    the Thule Expedition  ->  Dr. Mira Kovac  ->  the Halcyon Interferometer
                          ->  the Rowan Observatory  ->  Yellowknife

Around each chain sit a dozen near-duplicate documents that repeat the
expedition's name — departure dates, funding notes, leadership arrangements,
equipment logistics — restated over and over the way a real corpus accumulates
coverage of a notable entity. Two of those families are deliberately *adjacent*
to what the questions ask (leadership, equipment) without ever naming the leader
or the instrument, so they compete with the real evidence on topic and not just
on surface form.

In [ ]:
corpus = make_corpus(n_chains=20, seed=0, n_distractors=12)
print(corpus.describe())

chain0 = [d for d in corpus.documents if d.metadata["chain"] == 0]
print("\n--- the evidence chain -------------------------------------------")
for doc in chain0:
    if doc.metadata["role"] == "hop":
        print(f"  hop {doc.metadata['hop']}: {doc.text}")

print("\n--- a sample of the thicket around it ----------------------------")
for doc in [d for d in chain0 if d.metadata["role"] == "distractor"][:5]:
    print(f"  {doc.text}")

In [ ]:
for q in corpus.questions[:3]:
    print(f"{q.n_hops}-hop | {q.question}")
    print(f"        answer: {q.answer}   evidence: {q.gold_ids}\n")

## 2. Index it

`VendiRetriever` is the whole retriever: index documents, get a set back. It
takes any embedder with an `encode(texts) -> (n, d)` method.

We pin `HashingEmbedder` — pure numpy, deterministic, downloads nothing — so
this notebook produces the same numbers on every machine. One line swaps in a
real encoder:

```python
from vendirag import SentenceTransformerEmbedder
retriever = VendiRetriever(embedder=SentenceTransformerEmbedder("all-mpnet-base-v2"), ...)
```

A stronger encoder already handles some of the redundancy on its own, so the
gap between the two retrievers narrows — the shape of the result holds, the
magnitude shrinks.

In [ ]:
retriever = VendiRetriever(
    embedder=HashingEmbedder(),
    k=K,                 # documents to return
    s=0.8,               # diversity weight (the paper's default)
    candidate_pool=POOL, # |C|: the pool the greedy selection runs over
).index(corpus.documents)

print(f"{len(retriever)} documents indexed")

## 3. What plain similarity search returns

Pick a question where the two retrievers disagree, so the contrast is visible
rather than asserted. The aggregate numbers in section 5 cover all 60.

In [ ]:
def evidence_found(question, docs):
    return sum(d.id in set(question.gold_ids) for d in docs)

question = next(
    q for q in corpus.questions
    if not evidence_found(q, retriever.similarity_search(q.question, k=K))
    and evidence_found(q, retriever.retrieve(q.question, k=K, s=0.8))
)
print(question.question)
print(f"({question.n_hops} hops, answer: {question.answer})\n")

top_k = retriever.similarity_search(question.question, k=K)
for doc in top_k:
    mark = ">" if doc.id in set(question.gold_ids) else " "
    print(f"{mark} {doc.text[:92]}")

Eight documents. Two facts between them, and neither is the one we need.

The Vendi Score puts a number on that: it is the *effective number of unique
documents* in a set — 1 if they are all identical, `k` if they are mutually
orthogonal.

In [ ]:
def unique_docs(docs):
    return retriever.set_diversity([d.text for d in docs])

print(f"documents retrieved     : {len(top_k)}")
print(f"effectively unique      : {unique_docs(top_k):.1f}")
print(f"evidence chain recovered: {evidence_found(question, top_k)} of {len(question.gold_ids)}")

## 4. The same pool, selected by the Vendi Retrieval Score

Same candidate pool, same encoder, same budget. Only the selection rule changes:

$$\mathrm{VRS}(D) = s \cdot \widetilde{\mathrm{VS}}(D) + (1-s)\cdot\widetilde{\mathrm{SS}}(q, D)$$

Documents are added greedily, each step taking the one that maximizes the score
of the resulting *set*. `s` is the only knob: 0 is pure relevance, 1 is pure
diversity.

In [ ]:
diverse = retriever.retrieve(question.question, k=K, s=0.8)
for doc in diverse:
    mark = ">" if doc.id in set(question.gold_ids) else " "
    print(f"{mark} {doc.text[:92]}")

print(f"\neffectively unique      : {unique_docs(diverse):.1f}")
print(f"evidence chain recovered: {evidence_found(question, diverse)} of {len(question.gold_ids)}")

## 5. Across every question, and against the baselines

MMR is the right thing to compare against: it is the standard diversity-aware
selector, and it is also what the paper's strongest baseline uses. The
difference is that MMR penalizes similarity to the single nearest already-picked
document — a *pairwise* notion of novelty — where the Vendi Score measures the
rank structure of the whole set at once.

In [ ]:
def mmr_retrieve(question_text, lam):
    qe = retriever.embed_query(question_text)
    pool = np.argsort(-(retriever.embeddings @ qe))[:POOL]
    idx = mmr_select(retriever.embeddings[pool], qe, k=K, lambda_mult=lam)
    return [retriever.documents[pool[i]] for i in idx]


def score(retrieve_fn):
    found, unique = [], []
    for q in corpus.questions:
        docs = retrieve_fn(q.question)
        found.append(evidence_found(q, docs) > 0)
        unique.append(unique_docs(docs))
    return float(np.mean(found)), float(np.mean(unique))


methods = [("top-k similarity", lambda q: retriever.similarity_search(q, k=K))]
methods += [(f"MMR (lambda={lam})", lambda q, l=lam: mmr_retrieve(q, l))
            for lam in (0.3, 0.5, 0.7)]
methods += [(f"Vendi retrieval (s={s})",
             lambda q, s=s: retriever.retrieve(q, k=K, s=s))
            for s in (0.2, 0.4, 0.6, 0.8, 1.0)]

print(f"{'method':<28}{'evidence found':>16}{'unique docs':>14}")
for name, fn in methods:
    recall, unique = score(fn)
    print(f"{name:<28}{recall:>15.0%}{unique:>9.1f}/{K}")

Relevance alone almost never reaches the evidence — it spends the whole budget
inside the thicket. Any diversity-aware selection fixes that.

Note the optimum is **interior**. At `s = 1` relevance is ignored within the pool
and selection drifts to whatever is most different, which is off-topic, so recall
falls again. That is the trade-off the whole method is about.

In [ ]:
s_values = np.linspace(0, 1, 21)
recalls, uniques = zip(*[score(lambda q, s=s: retriever.retrieve(q, k=K, s=s))
                         for s in s_values])

fig, ax = plt.subplots(figsize=(7.5, 4))
ax.plot(s_values, recalls, "o-", color="#e8710a", label="evidence found")
ax.plot(s_values, np.array(uniques) / K, "o-", color="#1a73e8",
        label="set diversity (Vendi Score / k)")
ax.axvline(0.8, ls="--", color="#888", lw=1)
ax.annotate("paper default\ns = 0.8", (0.8, 0.12), xytext=(0.84, 0.12),
            fontsize=9, color="#666")
ax.set_xlabel("diversity weight  s")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.set_title("Relevance and diversity pull in opposite directions")
ax.grid(alpha=0.3)
plt.tight_layout()

## 6. How deep does the thicket have to be?

`n_distractors` controls how much redundant coverage surrounds each chain. It is
the parameter the whole failure mode depends on, so it is worth seeing where
relevance-only retrieval actually breaks — and how far the fix carries.

In [ ]:
depths = [2, 4, 8, 12, 16, 20, 24]
curves = {"top-k similarity": [], "MMR (lambda=0.5)": [], "Vendi (s=0.8)": []}

for depth in depths:
    c = make_corpus(n_chains=20, seed=0, n_distractors=depth)
    r = VendiRetriever(embedder=HashingEmbedder(), k=K, candidate_pool=POOL).index(c.documents)

    def hit(fn):
        return float(np.mean([
            any(d.id in set(q.gold_ids) for d in fn(q.question)) for q in c.questions
        ]))

    def mmr_here(question_text, lam=0.5):
        qe = r.embed_query(question_text)
        pool = np.argsort(-(r.embeddings @ qe))[:POOL]
        return [r.documents[pool[i]]
                for i in mmr_select(r.embeddings[pool], qe, k=K, lambda_mult=lam)]

    curves["top-k similarity"].append(hit(lambda q: r.similarity_search(q, k=K)))
    curves["MMR (lambda=0.5)"].append(hit(mmr_here))
    curves["Vendi (s=0.8)"].append(hit(lambda q: r.retrieve(q, k=K, s=0.8)))

fig, ax = plt.subplots(figsize=(7.5, 4))
for (name, values), colour in zip(curves.items(), ["#d93025", "#9aa0a6", "#1a73e8"]):
    ax.plot(depths, values, "o-", color=colour, label=name)
ax.set_xlabel("near-duplicate documents planted around each chain")
ax.set_ylabel("evidence found")
ax.set_ylim(0, 1.05)
ax.legend(frameon=False)
ax.set_title("Similarity search collapses; set-level selection degrades gracefully")
ax.grid(alpha=0.3)
plt.tight_layout()

With almost no redundancy, plain similarity search is fine — there is nothing to
be diverse about, and MMR's novelty penalty actively costs it something. By eight
near-duplicates per chain similarity search has collapsed to near zero and stays
there, while set-level selection is still finding the evidence four times out of
five.

Diversity-aware selection is not immune, though: past twenty near-duplicates
every method is losing ground, because with `k = 8` there is simply not enough
budget left after covering the thicket. Diversity buys a much better exchange
rate on that budget, not an exemption from it.

## 7. Retrieval is only half of it

One retrieval cannot answer a 3- or 4-hop question no matter how diverse it is.
The document naming the observatory does not mention the expedition, so nothing
about the original query can reach it. You have to *follow* the chain.

That is what the Vendi-RAG loop does (Algorithm 1): retrieve, reason, answer,
judge, adjust `s`, rewrite the query, repeat — returning the best answer over
all iterations.

`VendiRAG.offline` runs it with `HeuristicBackend`, a model-free stand-in that
walks the entity graph in the retrieved text instead of reasoning about it. It
does not understand language and it is not a substitute for an LLM — but the
control flow it exercises (the judge signal, the EMA on `s`, early stopping,
query refinement) is exactly the real thing, deterministically and for free.

In [ ]:
rag = VendiRAG.offline(
    retriever,
    k_docs=K, k_candidates=POOL,
    initial_s=0.8,       # s_1
    dynamic_s=False,     # fixed-s variant; see section 9
    max_iterations=5,    # N
    quality_threshold=0.85,  # tau
)

hard = next(q for q in corpus.questions if q.n_hops == 4)
result = rag.answer(hard.question)

print(f"Q: {hard.question}")
print(f"   gold: {hard.answer}\n")
for it in result.iterations:
    print(f"iteration {it.index}  s={it.s:.2f}  unique={it.vendi_score:.1f}/{K}  Q={it.quality:.2f}")
    print(f"  query : {it.query}")
    print(f"  answer: {it.answer}\n")
print(result)

Each iteration rewrites the query around the bridge entity it just reached, so
retrieval walks the chain one hop at a time. The loop stops as soon as the judge
clears `tau = 0.85`.

## 8. End to end

Four configurations, to separate what diversity contributes from what iteration
contributes.

In [ ]:
configs = [
    ("single-shot, top-k",        dict(initial_s=0.0), True),
    ("single-shot, Vendi s=0.8",  dict(initial_s=0.8), True),
    ("iterative, top-k",          dict(initial_s=0.0, dynamic_s=False), False),
    ("Vendi-RAG (fixed s=0.8)",   dict(initial_s=0.8, dynamic_s=False), False),
    ("Vendi-RAG (dynamic s)",     dict(initial_s=0.8, dynamic_s=True), False),
]

print(f"{'configuration':<28}{'exact match':>14}{'iterations':>13}")
outcomes = {}
for name, kwargs, single in configs:
    pipeline = VendiRAG.offline(retriever, k_docs=K, k_candidates=POOL, **kwargs)
    results = [
        pipeline.answer_single_shot(q.question) if single else pipeline.answer(q.question)
        for q in corpus.questions
    ]
    outcomes[name] = results
    em = np.mean([r.answer.strip().lower() == q.answer.strip().lower()
                  for r, q in zip(results, corpus.questions)])
    print(f"{name:<28}{em:>13.0%}{np.mean([r.n_iterations for r in results]):>13.1f}")

Neither half is sufficient alone. A single diverse retrieval cannot reach hop 3;
iterating over a redundant set just re-reads the same thicket. Together they
answer most of the chains.

## 9. What the dynamic `s` does, and when it helps

Between iterations the diversity weight moves by an exponential moving average
toward a target set by the judge:

$$s^{\text{target}} = \mathrm{clip}\!\left(1 - \frac{Q_t}{\max(\max_{\tau \le t} Q_\tau,\ \epsilon)},\ 0,\ 1\right),
\qquad s_{t+1} = \beta s_t + (1-\beta)\, s^{\text{target}}$$

Low quality raises the target toward diversity — explore for the evidence that
is missing. High quality lowers it toward relevance — consolidate along the
current reasoning path.

The target is **relative** to the running best, not absolutely calibrated, which
is what makes the loop robust to a smaller or different judge. It also has a
known consequence: when quality improves monotonically, every answer is the best
so far, so `s_target` is 0 at every step and `s` decays. On this corpus quality
is close to binary — the stub reader either closes the chain or does not — so
that is exactly what happens, and the dynamic variant lands slightly *behind*
fixed `s`.

That is consistent with the paper, which recommends the fixed-`s` configuration
as the default: `s` is the substantive knob, and adapting it online is a
second-order refinement whose gain concentrates on hard queries with a
well-graded judge.

In [ ]:
dynamic = outcomes["Vendi-RAG (dynamic s)"]
longest = max(dynamic, key=lambda r: r.n_iterations)

fig, ax = plt.subplots(figsize=(7.5, 3.6))
for r in dynamic[:14]:
    ax.plot(range(1, r.n_iterations + 1), r.s_trajectory,
            "o-", color="#1a73e8", alpha=0.35, lw=1.2, ms=4)
ax.plot(range(1, longest.n_iterations + 1), longest.s_trajectory,
        "o-", color="#e8710a", lw=2.4, ms=7, label="the longest trajectory")
ax.set_xlabel("iteration")
ax.set_ylabel("diversity weight  s")
ax.set_ylim(-0.05, 1.0)
ax.set_xticks(range(1, 6))
ax.legend(frameon=False)
ax.set_title("s trajectories under the EMA update")
ax.grid(alpha=0.3)
plt.tight_layout()

## 10. Using it for real

### With an LLM

The offline backend goes away and a real model takes over. Nothing else changes.

```python
from vendirag import VendiRAG, OpenAILLM       # pip install "vendirag[openai]"

rag = VendiRAG(retriever, llm=OpenAILLM("gpt-4o-mini"))
print(rag.answer("In which town is the observatory that houses it?"))
```

```python
from vendirag import AnthropicLLM              # pip install "vendirag[anthropic]"

rag = VendiRAG(retriever, llm=AnthropicLLM("claude-sonnet-5"))
```

A separate, cheaper judge is fine — the quality target is relative, not
calibrated:

```python
rag = VendiRAG(retriever,
               llm=AnthropicLLM("claude-sonnet-5"),
               judge_llm=OpenAILLM("gpt-4o-mini"))
```

Anything callable works too, so a local model or a LangChain runnable drops
straight in:

```python
rag = VendiRAG(retriever, llm=lambda prompt: my_local_model(prompt))
```

### With a vector store you already have

`VendiRetriever` keeps the corpus in memory, which is the wrong shape once it is
large. `VendiReranker` keeps your store and adds only the selection step.

In [ ]:
from vendirag import VendiReranker

# Stand in for a real store. In practice this queries Chroma / FAISS / pgvector
# and returns the pool it found, its vectors, and the encoded query.
def my_store(query, n):
    qe = retriever.embed_query(query)
    idx = np.argsort(-(retriever.embeddings @ qe))[:n]
    return [retriever.documents[i] for i in idx], retriever.embeddings[idx], qe


reranker = VendiReranker(my_store, s=0.8, k=K, candidate_pool=POOL)
for doc in reranker.retrieve(question.question)[:4]:
    print(f"- {doc.text[:88]}")

# The full loop runs against it unchanged.
print("\n", VendiRAG.offline(reranker, k_docs=K, k_candidates=POOL,
                             initial_s=0.8, dynamic_s=False).answer(hard.question))

### Just the score

The Vendi Score is useful on its own for measuring how redundant *any*
retriever's output is — including one that has nothing to do with this library.

In [ ]:
from vendirag import vendi_score

embedder = HashingEmbedder().fit(corpus.texts)
for label, docs in [("top-k similarity", top_k), ("Vendi retrieval", diverse)]:
    embeddings = embedder.encode([d.text for d in docs])
    print(f"{label:<20} {vendi_score(embeddings):.2f} effectively unique documents "
          f"out of {len(docs)}")

---

## Where to go next

- `vendirag.toy` — the corpus generator, if you want to change the shape of the trap
- `vendirag.viz.make_selection_gif` — the animation in the README
- `research/` — reproducing the paper's numbers on HotpotQA, MuSiQue, and 2WikiMultiHopQA
- The paper: [arXiv:2502.11228](https://arxiv.org/abs/2502.11228)